In [1]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Polygon, Point
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from collections import Counter
import os
import datetime as dt
import networkx as nx
import pickle
from itertools import product

In [3]:
coi_parques = gpd.read_file('../datos/9_pmts/COI_PARQUES_202504071925.geojson')

In [14]:
coi_parques.shape

(267, 30)

In [5]:
coi_parques.columns

Index(['OBJECTID', 'CID', 'DINI', 'DFIN', 'CONT', 'FINI', 'FFIN', 'HTRA',
       'HCIE', 'NCON', 'ASDM', 'LOCA', 'IRES', 'TAFE', 'PROR', 'REPR', 'VISI',
       'MASI', 'NCOM', 'FCOM', 'TCOI', 'RSDM', 'CREATED_USER', 'CREATED_DATE',
       'LAST_EDITED_USER', 'LAST_EDITED_DATE', 'OBSE', 'COMITE', 'FCOM_T',
       'geometry'],
      dtype='object')

In [32]:
coi_parques.OBSE.unique()

array(['NOTAS: 1. EL CONTRATISTA DEBE CONTAR CON LOS PERMISOS PERTINENTES PARA REALIZAR LAS ACTIVIDADES DE MANTENIMIENTO AL INTERIOR DEL PARQUE URBANIZACION COMPARTIR 11-260. 2. PREVIO A LA INTERVENCION SE DEBE SOCIALIZAR LAS ACTIVIDADES, EL PMT Y DEMAS RESTRICCIONES CON LA COMUNIDAD DIRECTAMENTE AFECTADA. 3. PARA TRABAJOS NOCTURNOS SE DEBE CONTAR CON LA AUTORIZACION POR PARTE DE LA ALCALDIA LOCAL DE ACUERDO A LO ESTIPULADO EN EL DECRETO 1076 DE 2015, ARTICULO 2.2.5.1.5.15 O AQUELLAS NORMAS QUE ADICIONEN, MODFIQUEN O SUSTITUYAN. 4. NO SE AUTORIZA INTERVENCION SOBRE FRANJAS PEATONALES PERIMETRALES DEL PARQUE.\n\nLA SDM AUTORIZA PMT PARA CIERRE DEL PARQUE (TUNA ALTA 11-1295) PARA ACTIVIDADES DE MANTENIMIENTO DEL PARQUE Y ACOPIO DE MATERIAL. IMPLEMENTAR LA SEÑALIZACION PROPUESTA Y MANTENERLA EN BUEN ESTADO. REALIZAR CERRAMIENTO CON DELINEADORES TUBULARES, TRIPLE CINTA Y/O POLISOMBRA TRASLUCIDA. NO SE DEBE PERMITIR EL ACCESO A LA ZONA DE CIERRE DE PERSONAL AJENO A LA OBRA. DEBE GARANTIZAR 

In [33]:
coi_puntos = gpd.read_file('../datos/9_pmts/COI_PUNTO_202504071914.geojson')

In [34]:
coi_puntos

,OBJECTID,CID,DINI,DFIN,CONT,FINI,FFIN,HTRA,HCIE,NCON,...,FCOM,TCOI,CREATED_USER,CREATED_DATE,LAST_EDITED_USER,LAST_EDITED_DATE,OBSE,COMITE,FCOM_T,geometry
0,283045,23602,AV CARACAS (CL 51 SUR) KR 9,N/A,CONSORCIO CARACAS SUR,2024-08-02 05:00:00+00:00,2024-09-05 05:00:00+00:00,24 HORAS,24 HORAS,IDU-1601-2019,...,2024-08-01 05:00:00+00:00,COI-ALTA,MSANCHEZM,2024-08-08 16:04:04+00:00,MSANCHEZM,2024-08-08 16:04:04+00:00,NOTA 1: EN CASO DE EXISTIR OBRAS DENTRO DEL AR...,31,1/08/2024,"LINESTRING (-74.12212 4.55781, -74.12212 4.55781)"
1,283046,23603,AV CARACAS (CL 51 SUR) KR 9,N/A,CONSORCIO CARACAS SUR,2024-08-02 05:00:00+00:00,2024-09-05 05:00:00+00:00,24 HORAS,24 HORAS,IDU-1601-2019,...,2024-08-01 05:00:00+00:00,COI-ALTA,MSANCHEZM,2024-08-08 16:04:04+00:00,MSANCHEZM,2024-08-08 16:04:04+00:00,NOTA 1: EN CASO DE EXISTIR OBRAS DENTRO DEL AR...,31,1/08/2024,"LINESTRING (-74.12224 4.55767, -74.12224 4.55767)"
2,283043,23600,AV CARACAS (CL 51 SUR) KR 9,N/A,CONSORCIO CARACAS SUR,2024-08-02 05:00:00+00:00,2024-09-05 05:00:00+00:00,24 HORAS,24 HORAS,IDU-1601-2019,...,2024-08-01 05:00:00+00:00,COI-ALTA,MSANCHEZM,2024-08-08 16:04:04+00:00,MSANCHEZM,2024-08-08 16:04:04+00:00,NOTA 1: EN CASO DE EXISTIR OBRAS DENTRO DEL AR...,31,1/08/2024,"LINESTRING (-74.1221 4.55783, -74.1221 4.55783)"
3,273703,2085,KR 65A No 98 06,N/A,JORGE MARQUEZ BOLIVAR,2023-01-20 05:00:00+00:00,2024-01-11 05:00:00+00:00,07:00-19:00,07:00-19:00,PRIVADO,...,2023-01-19 05:00:00+00:00,COI-BAJA,PMT,2024-08-02 22:15:20+00:00,PMT,2024-08-02 22:15:20+00:00,NOTAS: 1. DEBE CONTAR CON LA LICENCIA DE URBAN...,3,19/01/2023,"LINESTRING (-74.07025 4.68853, -74.07025 4.68853)"
4,273704,2086,KR 25 No 63B 20,N/A,JUAN CARLOS GONZALEZ CASTAÑO,2023-01-20 05:00:00+00:00,2024-01-11 05:00:00+00:00,07:00-18:00,07:00-18:00,PRIVADO,...,2023-01-19 05:00:00+00:00,COI-BAJA,PMT,2024-08-02 22:15:20+00:00,PMT,2024-08-02 22:15:20+00:00,NOTA: 1. DEBE CONTAR CON LA LICENCIA DE URBANI...,3,19/01/2023,"LINESTRING (-74.07277 4.65323, -74.07277 4.65323)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6902,290614,28864,AV DE LAS AMERICAS (AC 6) KR 69C,N/A,ENEL COLOMBIA S.A. E.S.P.,2024-09-27 05:00:00+00:00,2024-12-26 05:00:00+00:00,24 HORAS,24 HORAS,PRIVADO,...,2024-09-26 05:00:00+00:00,COI-BAJA,MSANCHEZM,2024-10-04 20:53:27+00:00,MSANCHEZM,2024-10-04 20:53:27+00:00,NOTAS: 1. EN CASO DE ENCONTRAR A OTRO CONTRATI...,39,26/09/2024,"LINESTRING (-74.12935 4.63001, -74.12935 4.63001)"
6903,290635,28839,KR 30A No 25B 27,N/A,CONSTRUCTORA EXPERTA S.A.S.,2024-09-27 05:00:00+00:00,2025-09-25 05:00:00+00:00,24 HORAS,24 HORAS,PRIVADO,...,2024-09-26 05:00:00+00:00,COI-BAJA,MSANCHEZM,2024-10-04 20:53:27+00:00,MSANCHEZM,2024-10-04 20:53:27+00:00,NOTAS: 1. EXTREMAR MEDIDAS DE SEGURIDAD DURANT...,39,26/09/2024,"LINESTRING (-74.08208 4.62683, -74.08208 4.62683)"
6904,290636,28892,KR 17A No 101 45,N/A,INGEURBE S.A.S.,2024-09-27 05:00:00+00:00,2025-09-25 05:00:00+00:00,07:00-18:00,N/A,PRIVADO,...,2024-09-26 05:00:00+00:00,COI-BAJA,MSANCHEZM,2024-10-04 20:53:27+00:00,MSANCHEZM,2024-10-04 20:53:27+00:00,NOTAS: 1. ESTE FRENTE DE OBRA NO CONTO CON AUT...,39,26/09/2024,"LINESTRING (-74.05023 4.68712, -74.05023 4.68712)"
6905,289311,27790,AC 100 AUTO NORTE (AK 45),N/A,CONSORCIO EUCARISTICO CARRERA 68,2024-09-13 05:00:00+00:00,2024-11-14 05:00:00+00:00,24 HORAS,24 HORAS,IDU-353-2020,...,2024-09-12 05:00:00+00:00,COI-BAJA,ARUIS,2024-09-24 03:48:24+00:00,ARUIS,2024-11-27 19:05:20+00:00,NOTA: ESTE FRENTE NO CONTO CON AUTORIZACION EN...,37,12/09/2024,"LINESTRING (-74.05642 4.68702, -74.05642 4.68702)"


In [35]:
coi_tramo = gpd.read_file('../datos/9_pmts/COI_TRAMO_202504071919.geojson')

In [37]:
coi_tramo.columns

Index(['OBJECTID', 'CID', 'DINI', 'DFIN', 'CONT', 'FINI', 'FFIN', 'HTRA',
       'HCIE', 'NCON', 'ASDM', 'LOCA', 'IRES', 'TAFE', 'PROR', 'REPR', 'VISI',
       'MASI', 'NCOM', 'FCOM', 'TCOI', 'RSDM', 'CREATED_USER', 'CREATED_DATE',
       'LAST_EDITED_USER', 'LAST_EDITED_DATE', 'OBSE', 'COMITE', 'FCOM_T',
       'ESTADO_VIA', 'geometry'],
      dtype='object')

In [39]:
coos_punto = gpd.read_file('../datos/9_pmts/COOS_PUNTO_202504071922.geojson')

In [40]:
coos_punto

,OBJECTID,CID,DINI,DFIN,CONT,ECON,HTRA,HCIE,LEXC,ASDM,...,FFIN,FCOM,CREATED_USER,OBSE,CREATED_DATE,LAST_EDITED_USER,LAST_EDITED_DATE,COMITE,FCOM_T,geometry
0,150572,4147,KR 80G No 6 19,N/A,DICO TELECOMUNICACIONES S.A.,TELECOMUNICACIONES,22:00-05:00,22:00-05:00,"IDU-032-2024, RES-405",SI,...,2024-10-31 05:00:00+00:00,2024-08-01 05:00:00+00:00,MSANCHEZM,NOTAS: 1. EN CASO DE ENCONTRAR OBRAS EN EL TRA...,2024-08-08 16:05:02+00:00,MSANCHEZM,2024-08-08 16:05:02+00:00,31,1/08/2024,"LINESTRING (-74.153 4.63627, -74.153 4.63627)"
1,161526,132,AK 68D No 13 97,N/A,EAAB E.S.P.,EAAB,09:00-17:00,09:00-17:00,IDU-018-2022,None,...,2024-12-27 05:00:00+00:00,2025-01-02 05:00:00+00:00,MSANCHEZM,NOTAS: 1. CUANDO SE AFECTE EL PARADERO DEL SIT...,2024-12-28 22:02:39+00:00,MSANCHEZM,2024-12-28 22:02:39+00:00,1,2/01/2025,"LINESTRING (-74.1212 4.64136, -74.1212 4.64136)"
2,156418,5598,CL 73 SUR KR 95A,N/A,DELTEC S.A.,ENEL-COLOMBIA,24 HORAS,24 HORAS,"IDU-199-2022, RES-4376",SI,...,2025-01-09 05:00:00+00:00,2024-10-24 05:00:00+00:00,MSANCHEZM,NOTAS: 1. EN CASO DE ENCONTRAR OBRAS EN EL TRA...,2024-10-28 18:45:40+00:00,MSANCHEZM,2024-10-28 18:45:40+00:00,43,24/10/2024,"LINESTRING (-74.20374 4.63095, -74.20374 4.63095)"
3,156104,5529,KR 100A CL 75A SUR,N/A,GRUPO DICON INGENIERIA S.A.S.,GAS NATURAL,06:00-18:00,06:00-18:00,"IDU-436-2019, RES-10083",SI,...,2025-01-02 05:00:00+00:00,2024-10-17 05:00:00+00:00,MSANCHEZM,NOTAS: 1. GARANTIZAR EN TODO MOMENTO EN CONDIC...,2024-10-19 20:09:08+00:00,MSANCHEZM,2024-10-19 20:09:08+00:00,42,17/10/2024,"LINESTRING (-74.20807 4.63331, -74.20807 4.63331)"
4,148578,7920,AV BOYACA (AK 72) AC 80,N/A,CONSORCIO ALIANZA 744,EAAB,22:00-05:00,22:00-05:00,IDU-155-2020,SI,...,2024-02-08 05:00:00+00:00,2023-12-21 05:00:00+00:00,PMT,"NOTAS: 1. PARA TRABAJOS NOCTURNOS, DOMINICALES...",2024-08-02 22:06:23+00:00,PMT,2024-08-02 22:06:23+00:00,51,21/12/2023,"LINESTRING (-74.08927 4.69653, -74.08927 4.69653)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2097,149721,3351,KR 87I 5m AL NORTE DE LA CL 57D SUR,N/A,ONNET FIBRA COLOMBIA S.A.S.,TELECOMUNICACIONES,07:00-17:00,24 HORAS,"IDU-181-2022, RES-4255",SI,...,2024-08-01 05:00:00+00:00,2024-06-06 05:00:00+00:00,PMT,NOTAS: 1. PARA TRABAJOS NOCTURNOS SE DEBE CONT...,2024-08-02 22:06:23+00:00,PMT,2024-08-02 22:06:23+00:00,23,6/06/2024,"LINESTRING (-74.18818 4.62529, -74.18818 4.62529)"
2098,149722,3364,DG 65G SUR KR 80K,N/A,ONNET FIBRA COLOMBIA S.A.S.,TELECOMUNICACIONES,22:00-05:00,22:00-05:00,"IDU-181-2022, RES-4255",SI,...,2024-08-01 05:00:00+00:00,2024-06-06 05:00:00+00:00,PMT,NOTAS: 1. PARA TRABAJOS NOCTURNOS SE DEBE CONT...,2024-08-02 22:06:23+00:00,PMT,2024-08-02 22:06:23+00:00,23,6/06/2024,"LINESTRING (-74.18886 4.6087, -74.18886 4.6087)"
2099,149723,3374,KR 81 No 74B 38 SUR,N/A,TECNOGAS LTDA.,GAS NATURAL,22:00-05:00,22:00-05:00,"IDU-226-2021, RES-3493",SI,...,2024-08-01 05:00:00+00:00,2024-06-06 05:00:00+00:00,PMT,"NOTAS: 1. PARA TRABAJOS NOCTURNOS, DOMINICALES...",2024-08-02 22:06:23+00:00,PMT,2024-08-02 22:06:23+00:00,23,6/06/2024,"LINESTRING (-74.20175 4.61138, -74.20175 4.61138)"
2100,149211,7432,CL 93A No 10 54,N/A,NICOLAS MANRIQUE CAMEJO - PROYECTO ONCE 93 S.A.S.,EAAB,07:00-19:00,07:00-19:00,"IDU-036-2022, RES-935",SI,...,2024-02-22 05:00:00+00:00,2023-11-30 05:00:00+00:00,PMT,NOTAS: 1. PREVIO A LA INTERVENCION SE DEBE SOC...,2024-08-02 22:06:23+00:00,PMT,2024-08-02 22:06:23+00:00,48,30/11/2023,"LINESTRING (-74.04666 4.67545, -74.04666 4.67545)"
